In [1]:
import gc
import time
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch


PROJECT_ROOT = Path(
    "/home/geonug/kdt-linux/ML"
)

LGBM_PATH = (
    PROJECT_ROOT
    / "models/network/lightgbm_top40_v4b2.txt"
)

BENCHMARK_DIR = (
    PROJECT_ROOT
    / "results/ondevice_benchmark_v4"
)

BENCHMARK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BATCH_SIZES = [
    1,
    32,
    256,
    4096,
]

BENCHMARK_ROWS = 100_000

WARMUP_RUNS = 20
MEASURE_RUNS = 100

print("PyTorch:", torch.__version__)
print("CPU threads:", torch.get_num_threads())

PyTorch: 2.14.1+cu130
CPU threads: 6


In [2]:
torch.set_num_threads(1)

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

print(
    "Torch threads:",
    torch.get_num_threads()
)

Torch threads: 1


In [4]:
from pathlib import Path
import gc
import time

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch


# ============================================================
# 1. Project paths
# ============================================================

PROJECT_ROOT = Path(
    "/home/geonug/kdt-linux/ML"
)

VALID_PATH = (
    PROJECT_ROOT
    / "data/processed/network_ml_dev_v4/validation/valid_v2_top40.parquet"
)

TOP40_PATH = (
    PROJECT_ROOT
    / "results/feature_selection/top40_features.csv"
)

SCALER_PATH = (
    PROJECT_ROOT
    / "models/network/dnn_top40_v4_scaler.joblib"
)

LGBM_PATH = (
    PROJECT_ROOT
    / "models/network/lightgbm_top40_v4b2.txt"
)

DNN_BEST_PATH = (
    PROJECT_ROOT
    / "models/network/dnn_top40_v4_best.pt"
)

BENCHMARK_DIR = (
    PROJECT_ROOT
    / "results/ondevice_benchmark_v4"
)

BENCHMARK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Benchmark settings
# ============================================================

BENCHMARK_ROWS = 100_000

BATCH_SIZES = [
    1,
    32,
    256,
    4096,
]

WARMUP_RUNS = 20
MEASURE_RUNS = 100


# ============================================================
# 3. Top40 feature list
# ============================================================

TOP40_FEATURES = (
    pd.read_csv(TOP40_PATH)["feature"]
    .astype(str)
    .tolist()
)


# ============================================================
# 4. StandardScaler
# ============================================================

scaler = joblib.load(
    SCALER_PATH
)


# ============================================================
# 5. Preprocessing
# ============================================================

def preprocess_batch(X, scaler):

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    X[~np.isfinite(X)] = np.nan

    means = scaler.mean_.astype(
        np.float32
    )

    rows, cols = np.where(
        np.isnan(X)
    )

    if len(rows) > 0:
        X[rows, cols] = means[cols]

    X = scaler.transform(
        X
    ).astype(
        np.float32,
        copy=False,
    )

    return X


# ============================================================
# 6. Path verification
# ============================================================

paths = {
    "VALID": VALID_PATH,
    "TOP40": TOP40_PATH,
    "SCALER": SCALER_PATH,
    "LIGHTGBM": LGBM_PATH,
    "DNN BEST": DNN_BEST_PATH,
}

print("=== PATH CHECK ===")

for name, path in paths.items():
    print(
        f"{name:10s}: "
        f"{'OK' if path.exists() else 'MISSING'} "
        f"{path}"
    )


print()
print(
    "Top40 features:",
    len(TOP40_FEATURES)
)

print(
    "Scaler features:",
    scaler.n_features_in_
)

print(
    "PyTorch:",
    torch.__version__
)

=== PATH CHECK ===
VALID     : OK /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v4/validation/valid_v2_top40.parquet
TOP40     : OK /home/geonug/kdt-linux/ML/results/feature_selection/top40_features.csv
SCALER    : OK /home/geonug/kdt-linux/ML/models/network/dnn_top40_v4_scaler.joblib
LIGHTGBM  : OK /home/geonug/kdt-linux/ML/models/network/lightgbm_top40_v4b2.txt
DNN BEST  : OK /home/geonug/kdt-linux/ML/models/network/dnn_top40_v4_best.pt

Top40 features: 40
Scaler features: 40
PyTorch: 2.14.1+cu130


In [5]:
pf = pq.ParquetFile(
    VALID_PATH
)

batch = next(
    pf.iter_batches(
        batch_size=BENCHMARK_ROWS,
        columns=TOP40_FEATURES,
    )
)

benchmark_df = (
    batch
    .to_pandas()
)


X_benchmark = (
    benchmark_df[
        TOP40_FEATURES
    ]
    .to_numpy(
        dtype=np.float32,
        copy=True,
    )
)


del benchmark_df, batch
gc.collect()


X_benchmark = preprocess_batch(
    X_benchmark,
    scaler,
)


assert X_benchmark.shape == (
    BENCHMARK_ROWS,
    40,
)

assert np.isfinite(
    X_benchmark
).all()


print(
    "Benchmark:",
    X_benchmark.shape
)

Benchmark: (100000, 40)


In [6]:
def benchmark_torch_model(
    model,
    X,
    batch_size,
    warmup_runs=20,
    measure_runs=100,
):

    model.eval()

    sample = (
        torch.from_numpy(
            X[:batch_size]
        )
    )


    # -------------------------
    # warm-up
    # -------------------------

    with torch.inference_mode():

        for _ in range(
            warmup_runs
        ):
            _ = model(sample)


    # -------------------------
    # measurement
    # -------------------------

    times_ms = []


    with torch.inference_mode():

        for _ in range(
            measure_runs
        ):

            start = (
                time.perf_counter_ns()
            )

            _ = model(sample)

            end = (
                time.perf_counter_ns()
            )


            elapsed_ms = (
                end - start
            ) / 1_000_000

            times_ms.append(
                elapsed_ms
            )


    times_ms = np.asarray(
        times_ms
    )


    median_ms = float(
        np.median(times_ms)
    )

    p95_ms = float(
        np.percentile(
            times_ms,
            95,
        )
    )


    throughput = (
        batch_size
        / (median_ms / 1000)
    )


    return {
        "batch_size":
            batch_size,

        "median_ms":
            median_ms,

        "p95_ms":
            p95_ms,

        "per_sample_us":
            (
                median_ms
                * 1000
                / batch_size
            ),

        "throughput_rows_sec":
            throughput,
    }

In [8]:
import torch
import torch.nn as nn
import lightgbm as lgb

from torchao.quantization import (
    Int8DynamicActivationInt8WeightConfig,
    quantize_,
)


# ============================================================
# 1. DNN architecture
# ============================================================

class NetworkThreatDNN(nn.Module):

    def __init__(
        self,
        input_dim=40,
        num_classes=15,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.network(x)


# ============================================================
# 2. CPU settings
# ============================================================

cpu_device = torch.device("cpu")

torch.set_num_threads(1)

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    # 이미 설정된 경우 발생할 수 있음
    pass


# ============================================================
# 3. Load checkpoint
# ============================================================

checkpoint = torch.load(
    DNN_BEST_PATH,
    map_location=cpu_device,
    weights_only=False,
)

print("Checkpoint loaded")

if "epoch" in checkpoint:
    print(
        "Best epoch:",
        checkpoint["epoch"]
    )


# ============================================================
# 4. Restore FP32
# ============================================================

fp32_cpu_model = NetworkThreatDNN().to(
    cpu_device
)

fp32_cpu_model.load_state_dict(
    checkpoint["model_state_dict"]
)

fp32_cpu_model.eval()

print("FP32 model restored")


# ============================================================
# 5. Create TorchAO INT8 from fresh FP32 weights
# ============================================================

torchao_int8_model = NetworkThreatDNN().to(
    cpu_device
)

torchao_int8_model.load_state_dict(
    checkpoint["model_state_dict"]
)

torchao_int8_model.eval()


int8_config = (
    Int8DynamicActivationInt8WeightConfig()
)

quantize_(
    torchao_int8_model,
    int8_config,
)

torchao_int8_model.eval()

print("TorchAO INT8 model created")


# ============================================================
# 6. LightGBM
# ============================================================

lgbm_model = lgb.Booster(
    model_file=str(LGBM_PATH)
)

print("LightGBM model loaded")


# ============================================================
# 7. Sanity check
# ============================================================

test_tensor = torch.from_numpy(
    X_benchmark[:32]
)


with torch.inference_mode():

    fp32_output = fp32_cpu_model(
        test_tensor
    )

    int8_output = torchao_int8_model(
        test_tensor
    )


fp32_test_pred = (
    fp32_output
    .argmax(dim=1)
    .numpy()
)

int8_test_pred = (
    int8_output
    .argmax(dim=1)
    .numpy()
)


print()
print("=== MODEL CHECK ===")

print(
    "FP32 output:",
    fp32_output.shape
)

print(
    "INT8 output:",
    int8_output.shape
)

print(
    "FP32 / INT8 agreement:",
    np.mean(
        fp32_test_pred
        == int8_test_pred
    )
)

print(
    "LightGBM trees:",
    lgbm_model.num_trees()
)

print(
    "Torch threads:",
    torch.get_num_threads()
)

W1006 14:30:34.808000 2151792 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Checkpoint loaded
Best epoch: 13
FP32 model restored
TorchAO INT8 model created
LightGBM model loaded

=== MODEL CHECK ===
FP32 output: torch.Size([32, 15])
INT8 output: torch.Size([32, 15])
FP32 / INT8 agreement: 1.0
LightGBM trees: 15000
Torch threads: 1


In [9]:
fp32_results = []


for batch_size in BATCH_SIZES:

    result = benchmark_torch_model(
        model=fp32_cpu_model,
        X=X_benchmark,
        batch_size=batch_size,
        warmup_runs=WARMUP_RUNS,
        measure_runs=MEASURE_RUNS,
    )

    result["model"] = (
        "DNN FP32"
    )

    fp32_results.append(
        result
    )

    print(
        batch_size,
        result
    )

1 {'batch_size': 1, 'median_ms': 0.047521, 'p95_ms': 0.12769625, 'per_sample_us': 47.521, 'throughput_rows_sec': 21043.328212790133, 'model': 'DNN FP32'}
32 {'batch_size': 32, 'median_ms': 0.0854375, 'p95_ms': 0.2946385999999999, 'per_sample_us': 2.669921875, 'throughput_rows_sec': 374542.7944403804, 'model': 'DNN FP32'}
256 {'batch_size': 256, 'median_ms': 0.21438000000000001, 'p95_ms': 0.37439409999999995, 'per_sample_us': 0.8374218750000001, 'throughput_rows_sec': 1194141.2445190782, 'model': 'DNN FP32'}
4096 {'batch_size': 4096, 'median_ms': 3.5562674999999997, 'p95_ms': 5.42808475, 'per_sample_us': 0.8682293701171874, 'throughput_rows_sec': 1151769.3761788167, 'model': 'DNN FP32'}


In [10]:
fp32_results = []

for batch_size in BATCH_SIZES:

    result = benchmark_torch_model(
        model=fp32_cpu_model,
        X=X_benchmark,
        batch_size=batch_size,
        warmup_runs=WARMUP_RUNS,
        measure_runs=MEASURE_RUNS,
    )

    result["model"] = "DNN FP32"

    fp32_results.append(result)

    print(
        f"batch={batch_size:4d} | "
        f"median={result['median_ms']:.4f} ms | "
        f"p95={result['p95_ms']:.4f} ms | "
        f"throughput={result['throughput_rows_sec']:,.0f} rows/s"
    )

batch=   1 | median=0.0374 ms | p95=0.0651 ms | throughput=26,761 rows/s
batch=  32 | median=0.0816 ms | p95=0.2156 ms | throughput=391,991 rows/s
batch= 256 | median=0.2090 ms | p95=0.2742 ms | throughput=1,225,168 rows/s
batch=4096 | median=3.2689 ms | p95=5.4692 ms | throughput=1,253,030 rows/s


In [12]:
def benchmark_lightgbm(
    model,
    X,
    batch_size,
    warmup_runs=20,
    measure_runs=100,
):

    sample = np.ascontiguousarray(
        X[:batch_size],
        dtype=np.float32,
    )

    # Warm-up
    for _ in range(warmup_runs):
        _ = model.predict(
            sample,
            num_threads=1,
        )

    times_ms = []

    # Measurement
    for _ in range(measure_runs):

        start = time.perf_counter_ns()

        _ = model.predict(
            sample,
            num_threads=1,
        )

        end = time.perf_counter_ns()

        times_ms.append(
            (end - start) / 1_000_000
        )

    times_ms = np.asarray(
        times_ms,
        dtype=np.float64,
    )

    median_ms = float(
        np.median(times_ms)
    )

    p95_ms = float(
        np.percentile(
            times_ms,
            95,
        )
    )

    return {
        "batch_size": batch_size,
        "median_ms": median_ms,
        "p95_ms": p95_ms,
        "per_sample_us": (
            median_ms
            * 1000
            / batch_size
        ),
        "throughput_rows_sec": (
            batch_size
            / (median_ms / 1000)
        ),
    }


print("benchmark_lightgbm defined")

benchmark_lightgbm defined


In [13]:
lgbm_results = []

print("=== LightGBM V4-B2 ===")

for batch_size in BATCH_SIZES:

    result = benchmark_lightgbm(
        model=lgbm_model,
        X=X_benchmark,
        batch_size=batch_size,
        warmup_runs=WARMUP_RUNS,
        measure_runs=MEASURE_RUNS,
    )

    result["model"] = "LightGBM V4-B2"

    lgbm_results.append(result)

    print(
        f"batch={batch_size:4d} | "
        f"median={result['median_ms']:.4f} ms | "
        f"p95={result['p95_ms']:.4f} ms | "
        f"throughput="
        f"{result['throughput_rows_sec']:,.0f} rows/s"
    )

=== LightGBM V4-B2 ===
batch=   1 | median=4.2867 ms | p95=6.9673 ms | throughput=233 rows/s
batch=  32 | median=58.2476 ms | p95=82.1416 ms | throughput=549 rows/s
batch= 256 | median=431.5961 ms | p95=533.4953 ms | throughput=593 rows/s
batch=4096 | median=7140.9082 ms | p95=8827.8094 ms | throughput=574 rows/s


In [11]:
# ============================================================
# TorchAO INT8
# ============================================================

int8_results = []

print("=== TorchAO INT8 ===")

for batch_size in BATCH_SIZES:

    result = benchmark_torch_model(
        model=torchao_int8_model,
        X=X_benchmark,
        batch_size=batch_size,
        warmup_runs=WARMUP_RUNS,
        measure_runs=MEASURE_RUNS,
    )

    result["model"] = "DNN TorchAO INT8"
    int8_results.append(result)

    print(
        f"batch={batch_size:4d} | "
        f"median={result['median_ms']:.4f} ms | "
        f"p95={result['p95_ms']:.4f} ms | "
        f"throughput={result['throughput_rows_sec']:,.0f} rows/s"
    )


# ============================================================
# LightGBM
# ============================================================

lgbm_results = []

print("\n=== LightGBM V4-B2 ===")

for batch_size in BATCH_SIZES:

    result = benchmark_lightgbm(
        model=lgbm_model,
        X=X_benchmark,
        batch_size=batch_size,
        warmup_runs=WARMUP_RUNS,
        measure_runs=MEASURE_RUNS,
    )

    result["model"] = "LightGBM V4-B2"
    lgbm_results.append(result)

    print(
        f"batch={batch_size:4d} | "
        f"median={result['median_ms']:.4f} ms | "
        f"p95={result['p95_ms']:.4f} ms | "
        f"throughput={result['throughput_rows_sec']:,.0f} rows/s"
    )

=== TorchAO INT8 ===
batch=   1 | median=0.5198 ms | p95=0.9398 ms | throughput=1,924 rows/s
batch=  32 | median=0.9835 ms | p95=1.9622 ms | throughput=32,538 rows/s
batch= 256 | median=4.4311 ms | p95=6.6938 ms | throughput=57,773 rows/s
batch=4096 | median=63.5359 ms | p95=71.4432 ms | throughput=64,468 rows/s

=== LightGBM V4-B2 ===


NameError: name 'benchmark_lightgbm' is not defined

In [14]:
FP32_SIZE_PATH = (
    BENCHMARK_DIR
    / "dnn_fp32_state_dict.pt"
)

INT8_SIZE_PATH = (
    BENCHMARK_DIR
    / "dnn_torchao_int8_state_dict.pt"
)

torch.save(
    fp32_cpu_model.state_dict(),
    FP32_SIZE_PATH,
)

torch.save(
    torchao_int8_model.state_dict(),
    INT8_SIZE_PATH,
)

print("=== MODEL SIZE ===")

print(
    f"LightGBM V4-B2   : "
    f"{LGBM_PATH.stat().st_size / 1024**2:.4f} MB"
)

print(
    f"DNN FP32         : "
    f"{FP32_SIZE_PATH.stat().st_size / 1024**2:.4f} MB"
)

print(
    f"DNN TorchAO INT8 : "
    f"{INT8_SIZE_PATH.stat().st_size / 1024**2:.4f} MB"
)

=== MODEL SIZE ===
LightGBM V4-B2   : 46.3843 MB
DNN FP32         : 0.1286 MB
DNN TorchAO INT8 : 0.0459 MB


In [15]:
import psutil

print("psutil:", psutil.__version__)

psutil: 7.2.2
